# RHS2116 SPI IP Hardware Verification Tests (T1-T4)

This notebook tests the RHS2116 SPI IP on KV260.

## Setup - Import Libraries

In [1]:
from pynq import Overlay, MMIO
from pynq import allocate
import numpy as np
import time

## Register Address Definitions

In [2]:
# Configuration Registers (R/W)
ADDR_CTRL_REG       = 0x00
ADDR_MAX_TIMESTEP   = 0x04
ADDR_DATACLK_CFG    = 0x08
ADDR_MISO_DELAY     = 0x0C
ADDR_DATA_STREAM_EN = 0x10

# Trigger Register
ADDR_TRIGGER_REG    = 0x80

# Status Registers (R)
ADDR_STATUS_REG     = 0xA0
ADDR_FIFO_STATUS    = 0xA4
ADDR_TIMESTAMP      = 0xA8
ADDR_BOARD_ID       = 0xB0
ADDR_BOARD_VERSION  = 0xB4

# Trigger bits
TRIG_SPI_START      = 0x01
TRIG_RESET_SEQ      = 0x02

---
## T1: Overlay Loading Test

In [3]:
print('=' * 50)
print('T1: Overlay Loading Test')
print('=' * 50)

ol = Overlay('/root/jupyter_notebooks/RHS_SPI/rhs2116_system.bit')
print('[PASS] Overlay loaded successfully!')
print(f'IP blocks: {list(ol.ip_dict.keys())}')

# Show address mapping
print('\nAddress mapping:')
for name, info in ol.ip_dict.items():
    print(f'  {name}: 0x{info["phys_addr"]:08X}')

T1: Overlay Loading Test


[PASS] Overlay loaded successfully!
IP blocks: ['rhs2000_spi_ip_0', 'axi_dma_0', 'zynq_ultra_ps_e_0']

Address mapping:
  rhs2000_spi_ip_0: 0xA0000000
  axi_dma_0: 0xA0010000


KeyError: 'phys_addr'

---
## T2: Register Read/Write Test

In [ ]:
print('=' * 50)
print('T2: Register Read/Write Test')
print('=' * 50)

# Get RHS IP base address
rhs_ip = None
for name, info in ol.ip_dict.items():
    if 'rhs' in name.lower():
        rhs_ip = info
        break

if rhs_ip:
    base_addr = rhs_ip['phys_addr']
    print(f'Found RHS IP at: 0x{base_addr:08X}')
else:
    base_addr = 0xA0000000
    print(f'Using default base address: 0x{base_addr:08X}')

rhs = MMIO(base_addr, 0x1000)

In [ ]:
# T2.1: Read BOARD_ID
print('[T2.1] Reading BOARD_ID...')
board_id = rhs.read(ADDR_BOARD_ID)
expected = 900
status = 'PASS' if board_id == expected else 'FAIL'
print(f'  BOARD_ID = {board_id} (expected: {expected}) [{status}]')

In [ ]:
# T2.2: Read BOARD_VERSION
print('[T2.2] Reading BOARD_VERSION...')
board_ver = rhs.read(ADDR_BOARD_VERSION)
expected = 1
status = 'PASS' if board_ver == expected else 'FAIL'
print(f'  BOARD_VERSION = {board_ver} (expected: {expected}) [{status}]')

In [ ]:
# T2.3: Write/Read MAX_TIMESTEP
print('[T2.3] Testing MAX_TIMESTEP write/read...')
test_val = 0x12345678
rhs.write(ADDR_MAX_TIMESTEP, test_val)
readback = rhs.read(ADDR_MAX_TIMESTEP)
status = 'PASS' if readback == test_val else 'FAIL'
print(f'  Wrote: 0x{test_val:08X}, Read: 0x{readback:08X} [{status}]')

In [ ]:
# T2.4: Write/Read MISO_DELAY
print('[T2.4] Testing MISO_DELAY write/read...')
test_val = 0x05
rhs.write(ADDR_MISO_DELAY, test_val)
readback = rhs.read(ADDR_MISO_DELAY) & 0x0F
status = 'PASS' if readback == test_val else 'FAIL'
print(f'  Wrote: 0x{test_val:02X}, Read: 0x{readback:02X} [{status}]')

In [ ]:
# T2.5: Read STATUS_REG
print('[T2.5] Reading STATUS_REG...')
status_reg = rhs.read(ADDR_STATUS_REG)
spi_running = status_reg & 0x01
dataclk_locked = (status_reg >> 1) & 0x01
print(f'  STATUS_REG = 0x{status_reg:08X}')
print(f'    spi_running = {spi_running}')
print(f'    dataclk_locked = {dataclk_locked}')

In [ ]:
# T2.6: Read TIMESTAMP
print('[T2.6] Reading TIMESTAMP...')
timestamp = rhs.read(ADDR_TIMESTAMP)
print(f'  TIMESTAMP = {timestamp}')

---
## T3: SPI Signal Output Test

**NOTE:** Use oscilloscope to verify PMOD signals:
- Pin 5 (D10): SCLK
- Pin 7 (C11): CS_n
- Pin 3 (E10): MOSI
- Pin 2 (B10): spi_running

In [ ]:
print('=' * 50)
print('T3: SPI Signal Output Test')
print('=' * 50)

# Reset and configure
print('[T3.1] Configuring SPI...')
rhs.write(ADDR_CTRL_REG, 0x00)  # Stop SPI
rhs.write(ADDR_MAX_TIMESTEP, 100)  # Run 100 timesteps
rhs.write(ADDR_MISO_DELAY, 0x05)  # Set delay

# Check initial status
status = rhs.read(ADDR_STATUS_REG)
print(f'  Initial STATUS: 0x{status:08X} (spi_running = {status & 1})')

In [ ]:
# Start SPI
print('[T3.2] Starting SPI (single shot)...')
rhs.write(ADDR_TRIGGER_REG, TRIG_SPI_START)

time.sleep(0.01)  # Small delay

# Check status
status = rhs.read(ADDR_STATUS_REG)
spi_running = status & 0x01
timestamp = rhs.read(ADDR_TIMESTAMP)
print(f'  STATUS after trigger: 0x{status:08X}')
print(f'    spi_running = {spi_running}')
print(f'    timestamp = {timestamp}')

print('\n>>> Check oscilloscope for SPI signals <<<')

In [ ]:
# Wait for completion
print('[T3.3] Waiting for SPI to complete...')
time.sleep(0.5)

status = rhs.read(ADDR_STATUS_REG)
spi_running = status & 0x01
timestamp = rhs.read(ADDR_TIMESTAMP)
print(f'  Final STATUS: 0x{status:08X}')
print(f'    spi_running = {spi_running}')
print(f'    timestamp = {timestamp}')

---
## T4: DMA Data Transfer Test

In [ ]:
print('=' * 50)
print('T4: DMA Data Transfer Test')
print('=' * 50)

# Get DMA
try:
    dma = ol.axi_dma_0
    print(f'[INFO] DMA found: {dma}')
    print(f'  Recv channel: {dma.recvchannel}')
except Exception as e:
    print(f'[FAIL] Cannot get DMA: {e}')

In [ ]:
# Parameters
NUM_SAMPLES = 10
BYTES_PER_SAMPLE = 112  # 14 beats * 8 bytes

# T4.1: Allocate buffer
print('[T4.1] Allocating DMA buffer...')
buffer_size = NUM_SAMPLES * BYTES_PER_SAMPLE
recv_buffer = allocate(shape=(buffer_size // 8,), dtype=np.uint64)
print(f'  Allocated {buffer_size} bytes at 0x{recv_buffer.physical_address:08X}')

In [ ]:
# T4.2: Configure SPI
print('[T4.2] Configuring SPI...')
rhs.write(ADDR_CTRL_REG, 0x00)
rhs.write(ADDR_MAX_TIMESTEP, NUM_SAMPLES)
print(f'  MAX_TIMESTEP = {NUM_SAMPLES}')

In [ ]:
# T4.3: Start DMA and SPI
print('[T4.3] Starting DMA receive and SPI...')

dma.recvchannel.transfer(recv_buffer)
rhs.write(ADDR_TRIGGER_REG, TRIG_SPI_START)

# Wait for completion
try:
    dma.recvchannel.wait(timeout=5)
    print('  DMA transfer completed!')
except Exception as e:
    print(f'  DMA timeout or error: {e}')

In [ ]:
# T4.4 & T4.5: Verify data
print('[T4.4] Verifying Magic Number...')
print('[T4.5] Checking Timestamp sequence...')

for i in range(min(NUM_SAMPLES, 5)):
    offset = i * (BYTES_PER_SAMPLE // 8)
    beat0 = recv_buffer[offset]
    magic = beat0 & 0xFFFFFFFF
    timestamp = (beat0 >> 32) & 0xFFFFFFFF
    
    magic_ok = (magic == 0x21160001)
    status = 'PASS' if magic_ok else 'FAIL'
    print(f'  Sample {i}: Magic=0x{magic:08X} [{status}], Timestamp={timestamp}')

In [ ]:
# Cleanup
del recv_buffer
print('\nBuffer released.')

---
## Test Summary

| Test | Description | Status |
|------|-------------|--------|
| T1 | Overlay Loading | |
| T2.1 | BOARD_ID = 900 | |
| T2.2 | BOARD_VERSION = 1 | |
| T2.3 | MAX_TIMESTEP R/W | |
| T2.4 | MISO_DELAY R/W | |
| T3 | SPI Output (manual) | |
| T4.4 | Magic = 0x21160001 | |
| T4.5 | Timestamp sequence | |